# Gold Layer — Supply Chain Data Platform

Aggregates and enriches Silver tables into business-ready, analytics-optimised Gold tables.

| Silver Sources | Gold Target | Key Transformations |
|---|---|---|
| silver_fact_orders + silver_dim_products + silver_dim_warehouses | gold_order_summary | Denormalised wide fact table; line_revenue = quantity × unit_price |
| silver_fact_shipments + silver_fact_orders | gold_shipment_performance | Delivery lead days; on-time flag (≤ 7 days); cost per unit |
| silver_dim_products + silver_fact_orders | gold_product_revenue | Aggregated revenue, quantity, order counts per product |
| silver_dim_warehouses + orders + shipments | gold_warehouse_operations | Warehouse throughput, revenue, on-time delivery % |

In [0]:
-- Create the Gold schema if it doesn't already exist
CREATE SCHEMA IF NOT EXISTS abd_supplychain_dev.gold;

In [0]:
-- Gold: gold_order_summary
-- Wide order fact table: orders denormalised with product and warehouse dimensions
-- Cleaned: inner-joins ensure only matched records (referential integrity)
-- Derived: line_revenue_usd = quantity * unit_price_usd
-- Derived: is_fulfilled flag (order_status = 'DELIVERED')
CREATE OR REPLACE TABLE abd_supplychain_dev.gold.gold_order_summary AS
SELECT
  o.order_id,
  o.customer_id,
  o.order_date,
  o.order_timestamp,
  o.order_status,
  CASE WHEN o.order_status = 'DELIVERED' THEN TRUE ELSE FALSE END  AS is_fulfilled,
  -- Product dimension
  p.product_id,
  p.sku,
  p.category                              AS product_category,
  p.unit_price_usd,
  p.price_tier,
  -- Warehouse dimension
  w.warehouse_id,
  w.warehouse_name,
  w.location                              AS warehouse_location,
  -- Metrics
  o.quantity,
  ROUND(o.quantity * p.unit_price_usd, 2) AS line_revenue_usd,
  current_timestamp()                     AS _gold_processed_at
FROM abd_supplychain_dev.silver.silver_fact_orders          o
INNER JOIN abd_supplychain_dev.silver.silver_dim_products   p  ON o.product_id   = p.product_id
INNER JOIN abd_supplychain_dev.silver.silver_dim_warehouses w  ON o.warehouse_id = w.warehouse_id;

In [0]:
-- Gold: gold_shipment_performance
-- Joins shipments with orders to surface delivery KPIs
-- Cleaned: only shipments with a matching order are included
-- Derived: delivery_lead_days = DATEDIFF(ship_date, order_date)
-- Derived: is_on_time (delivery_lead_days <= 7)
-- Derived: cost_per_unit_usd = shipment_cost / quantity
CREATE OR REPLACE TABLE abd_supplychain_dev.gold.gold_shipment_performance AS
SELECT
  s.shipment_id,
  s.order_id,
  s.carrier,
  s.ship_date,
  s.ship_timestamp,
  s.shipment_cost_usd,
  s.delivery_notes,
  -- From order
  o.customer_id,
  o.product_id,
  o.warehouse_id,
  o.quantity,
  o.order_date,
  o.order_status,
  -- Delivery KPIs
  DATEDIFF(s.ship_date, o.order_date)                                AS delivery_lead_days,
  CASE
    WHEN DATEDIFF(s.ship_date, o.order_date) <= 7 THEN TRUE
    ELSE FALSE
  END                                                                AS is_on_time,
  CASE
    WHEN o.quantity > 0 THEN ROUND(s.shipment_cost_usd / o.quantity, 4)
    ELSE NULL
  END                                                                AS cost_per_unit_usd,
  current_timestamp()                                                AS _gold_processed_at
FROM abd_supplychain_dev.silver.silver_fact_shipments               s
INNER JOIN abd_supplychain_dev.silver.silver_fact_orders            o  ON s.order_id = o.order_id;

In [0]:
-- Gold: gold_product_revenue
-- Aggregated product-level KPIs: revenue, volume, and fulfilment counts
-- Grain: one row per product
CREATE OR REPLACE TABLE abd_supplychain_dev.gold.gold_product_revenue AS
SELECT
  p.product_id,
  p.sku,
  p.category,
  p.price_tier,
  p.unit_price_usd,
  COUNT(DISTINCT o.order_id)                                                           AS total_orders,
  SUM(o.quantity)                                                                      AS total_quantity_sold,
  ROUND(SUM(o.quantity * p.unit_price_usd), 2)                                         AS total_revenue_usd,
  ROUND(AVG(o.quantity * p.unit_price_usd), 2)                                         AS avg_revenue_per_order_usd,
  ROUND(AVG(o.quantity), 2)                                                            AS avg_quantity_per_order,
  COUNT(DISTINCT CASE WHEN o.order_status = 'DELIVERED' THEN o.order_id END)           AS delivered_orders,
  current_timestamp()                                                                  AS _gold_processed_at
FROM abd_supplychain_dev.silver.silver_dim_products                 p
LEFT JOIN abd_supplychain_dev.silver.silver_fact_orders             o  ON p.product_id = o.product_id
GROUP BY p.product_id, p.sku, p.category, p.price_tier, p.unit_price_usd;

In [0]:
-- Gold: gold_warehouse_operations
-- Aggregated warehouse-level KPIs: throughput, revenue, on-time delivery %
-- Grain: one row per warehouse
CREATE OR REPLACE TABLE abd_supplychain_dev.gold.gold_warehouse_operations AS
WITH shipment_kpis AS (
  SELECT
    s.shipment_id,
    s.order_id,
    s.shipment_cost_usd,
    CASE
      WHEN DATEDIFF(s.ship_date, o.order_date) <= 7 THEN 1
      ELSE 0
    END AS is_on_time
  FROM abd_supplychain_dev.silver.silver_fact_shipments  s
  INNER JOIN abd_supplychain_dev.silver.silver_fact_orders o ON s.order_id = o.order_id
)
SELECT
  w.warehouse_id,
  w.warehouse_name,
  w.location,
  w.max_capacity_units,
  COUNT(DISTINCT o.order_id)                                              AS total_orders,
  SUM(o.quantity)                                                         AS total_units_processed,
  ROUND(SUM(o.quantity * p.unit_price_usd), 2)                           AS total_revenue_usd,
  ROUND(AVG(o.quantity * p.unit_price_usd), 2)                           AS avg_order_value_usd,
  COUNT(DISTINCT sk.shipment_id)                                          AS total_shipments,
  ROUND(AVG(sk.shipment_cost_usd), 2)                                     AS avg_shipment_cost_usd,
  ROUND(
    SUM(sk.is_on_time) * 100.0 / NULLIF(COUNT(sk.shipment_id), 0), 2
  )                                                                       AS on_time_delivery_pct,
  current_timestamp()                                                     AS _gold_processed_at
FROM abd_supplychain_dev.silver.silver_dim_warehouses                    w
LEFT JOIN abd_supplychain_dev.silver.silver_fact_orders                  o   ON w.warehouse_id = o.warehouse_id
LEFT JOIN abd_supplychain_dev.silver.silver_dim_products                 p   ON o.product_id   = p.product_id
LEFT JOIN shipment_kpis                                                  sk  ON o.order_id     = sk.order_id
GROUP BY w.warehouse_id, w.warehouse_name, w.location, w.max_capacity_units;

In [0]:
-- Validation: confirm all Gold tables were populated
SELECT 'gold_order_summary'        AS table_name, COUNT(*) AS row_count FROM abd_supplychain_dev.gold.gold_order_summary
UNION ALL
SELECT 'gold_shipment_performance' AS table_name, COUNT(*) AS row_count FROM abd_supplychain_dev.gold.gold_shipment_performance
UNION ALL
SELECT 'gold_product_revenue'      AS table_name, COUNT(*) AS row_count FROM abd_supplychain_dev.gold.gold_product_revenue
UNION ALL
SELECT 'gold_warehouse_operations' AS table_name, COUNT(*) AS row_count FROM abd_supplychain_dev.gold.gold_warehouse_operations
ORDER BY table_name;